# Project 3: Deep Learning
## dataset used: Churn Modelling

In [1]:
import pandas as pd
import numpy as np
# =====================================================================
df = pd.read_csv("Churn_Modelling_with_nulls.csv")

df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602.0,Hargrave,619.0,France,Female,NaN,2.0,0.00,1.0,1.0,1.0,101348.88,1.0
1,2,15647311.0,Hill,608.0,Spain,Female,41.0,1.0,83807.86,1.0,0.0,1.0,112542.58,0.0
2,3,15619304.0,Onio,502.0,France,Female,42.0,8.0,159660.80,3.0,1.0,0.0,113931.57,1.0
3,4,15701354.0,Boni,699.0,France,Female,39.0,1.0,0.00,2.0,0.0,0.0,93826.63,0.0
4,5,15737888.0,Mitchell,850.0,Spain,Female,43.0,NaN,125510.82,1.0,1.0,1.0,79084.10,0.0


In [2]:
df.shape

(10000, 14)

In [3]:
df = df.dropna(subset=['Exited']).copy()
df['Exited'] = df['Exited'].astype(int)

In [4]:
df = df.drop(columns=['RowNumber', 'CustomerId', 'Surname'])

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9522 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CreditScore      9034 non-null   float64
 1   Geography        9042 non-null   object 
 2   Gender           9063 non-null   object 
 3   Age              9028 non-null   float64
 4   Tenure           9032 non-null   float64
 5   Balance          9057 non-null   float64
 6   NumOfProducts    9061 non-null   float64
 7   HasCrCard        9036 non-null   float64
 8   IsActiveMember   9047 non-null   float64
 9   EstimatedSalary  9069 non-null   float64
 10  Exited           9522 non-null   int64  
dtypes: float64(8), int64(1), object(2)
memory usage: 892.7+ KB


In [6]:
numerical_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 
                  'HasCrCard', 'IsActiveMember', 'EstimatedSalary']
categorical_cols = ['Geography', 'Gender']

In [7]:
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

In [8]:
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True, dtype=int)

In [9]:
X = df_encoded.drop(columns=['Exited'])
y = df_encoded['Exited']

In [10]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [11]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [12]:
neg, pos = np.bincount(y_train)
class_weight = {0: (1.0 / neg) * (len(y_train) / 2.0), 1: (1.0 / pos) * (len(y_train) / 2.0)}

In [15]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
model = Sequential([
    # Input Layer + First Hidden Layer
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.2),
    
    # Second Hidden Layer
    Dense(32, activation='relu'),
    Dropout(0.2),
    
    # Third Hidden Layer
    Dense(16, activation='relu'),
    
    # Output Layer (Sigmoid activation yields probabilities between 0 and 1)
    Dense(1, activation='sigmoid')
])

In [16]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

In [18]:
from tensorflow.keras.callbacks import EarlyStopping
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [19]:
print("--- Training Customer Churn Deep Learning Model ---")
history = model.fit(
    X_train_scaled, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    class_weight=class_weight,
    callbacks=[early_stop],
    verbose=1
)

--- Training Customer Churn Deep Learning Model ---
Epoch 1/50
191/191 ━━━━━━━━━━━━━━━━━━━━ 9s 14ms/step - accuracy: 0.6120 - auc: 0.4942 - loss: 0.6985 - val_accuracy: 0.1870 - val_auc: 0.5000 - val_loss: 0.6971
Epoch 2/50
191/191 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.2061 - auc: 0.5000 - loss: 0.6980 - val_accuracy: 0.1870 - val_auc: 0.5000 - val_loss: 0.7001
Epoch 3/50
191/191 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.3470 - auc: 0.4946 - loss: 0.6981 - val_accuracy: 0.1870 - val_auc: 0.5000 - val_loss: 0.7020
Epoch 4/50
191/191 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.2061 - auc: 0.5000 - loss: 0.6980 - val_accuracy: 0.1870 - val_auc: 0.5000 - val_loss: 0.7017
Epoch 5/50
191/191 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.2061 - auc: 0.4911 - loss: 0.6980 - val_accuracy: 0.1870 - val_auc: 0.5000 - val_loss: 0.7011
Epoch 6/50
191/191 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.2061 - auc: 0.4954 - loss: 0.6980 - val_accuracy: 0.1870 - val_auc: 0.5000 - va

In [20]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
# Generate Predictions
y_pred_proba = model.predict(X_test_scaled).flatten()
y_pred = (y_pred_proba > 0.5).astype(int)

# Final Printout Evaluation
print("\n" + "="*45)
print("🏆 DEEP LEARNING MODEL PROJECT RESULTS 🏆")
print("="*45)
print(classification_report(y_test, y_pred))
print(f"ROC AUC Score: {roc_auc_score(y_test, y_pred_proba):.4f}")

60/60 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step

🏆 DEEP LEARNING MODEL PROJECT RESULTS 🏆
              precision    recall  f1-score   support

           0       0.00      0.00      0.00      1519
           1       0.20      1.00      0.34       386

    accuracy                           0.20      1905
   macro avg       0.10      0.50      0.17      1905
weighted avg       0.04      0.20      0.07      1905

ROC AUC Score: 0.5000


C:\Program Files\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Program Files\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Program Files\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
